# device check

In [13]:
from pathlib import Path
import json

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights,resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


# path

In [14]:
DATA_ROOT = Path(r"D:\dataset\sampled_500")

TRAIN_DIR = DATA_ROOT / "train_mini"
VAL_DIR = DATA_ROOT / "validation"

MODEL_DIR = DATA_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

# MODEL_PATH = MODEL_DIR / "inat_resnet50_best.pth"

#processing

In [15]:
INPUT_SIZE = 320

# transforms.RandomResizedCrop(
#         350,
#         scale=(0.80, 1.0),       # at lease 80%
#         ratio=(0.85, 1.15),      # ratio limit
#         interpolation=transforms.InterpolationMode.BILINEAR,
#         antialias=True
#     ),

# transforms.Resize(INPUT_SIZE , interpolation=transforms.InterpolationMode.BILINEAR,
#        antialias=True),

train_transform = transforms.Compose([
    transforms.CenterCrop(INPUT_SIZE),
    
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# load traning and validation

In [16]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=val_transform
)

assert train_dataset.class_to_idx == val_dataset.class_to_idx, (
    "Train and validation folder differed"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,  
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=8,
    pin_memory=torch.cuda.is_available()
)

print("Classes:", len(train_dataset.classes))
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Class mapping:", train_dataset.class_to_idx)

Classes: 500
Train images: 20000
Validation images: 5000
Class mapping: {'00001_Animalia_Annelida_Polychaeta_Sabellida_Sabellidae_Sabella_spallanzanii': 0, '00003_Animalia_Annelida_Polychaeta_Sabellida_Serpulidae_Spirobranchus_cariniferus': 1, '00018_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Argiope_bruennichi': 2, '00024_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Cyclosa_turbinata': 3, '00038_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Neoscona_crucifera': 4, '00055_Animalia_Arthropoda_Arachnida_Araneae_Filistatidae_Kukulcania_hibernalis': 5, '00128_Animalia_Arthropoda_Arachnida_Araneae_Thomisidae_Synema_globosum': 6, '00145_Animalia_Arthropoda_Arachnida_Opiliones_Phalangiidae_Phalangium_opilio': 7, '00159_Animalia_Arthropoda_Chilopoda_Scolopendromorpha_Scolopendridae_Scolopendra_heros': 8, '00203_Animalia_Arthropoda_Insecta_Coleoptera_Carabidae_Cicindela_hirticollis': 9, '00216_Animalia_Arthropoda_Insecta_Coleoptera_Carabidae_Scaphinotus_angusticollis': 10, '00230_Anim

# training function

In [ ]:
def training(
    EPOCHS,
    model,
    optimizer,
    scheduler,
    criterion,
    fname,
    classes
):
    best_val_accuracy = -1.0
    early_stop_patience = 7
    epochs_without_improvement = 0
    min_improvement = 1e-4

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    # save training
    history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rates": []
    }
    history_fname = str(Path(fname).with_suffix("")) + "_history.json"

    for epoch in range(EPOCHS):
        #  Training 
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * labels.size(0)
            train_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            train_total += labels.size(0)

        #  Validation 
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.inference_mode():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=use_amp
                ):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * labels.size(0)
                val_correct += (
                    outputs.argmax(dim=1) == labels
                ).sum().item()
                val_total += labels.size(0)

        train_loss /= train_total
        val_loss /= val_total
        train_accuracy = train_correct / train_total
        val_accuracy = val_correct / val_total

        scheduler.step(val_accuracy)

        learning_rates = [
            f"{group['lr']:.2e}"
            for group in optimizer.param_groups
        ]

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train acc: {train_accuracy:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val acc: {val_accuracy:.4f} | "
            f"LR: {learning_rates}"
        )
        
        # data adding
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        history["learning_rates"].append(
            [group["lr"] for group in optimizer.param_groups]
        )
        with open(history_fname, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=4)

        if val_accuracy > best_val_accuracy + min_improvement:
            best_val_accuracy = val_accuracy
            epochs_without_improvement = 0

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "classes": classes,
                "num_classes": len(classes),
                "best_val_accuracy": best_val_accuracy,
                "train_accuracy": train_accuracy,
                "val_accuracy": val_accuracy,
                "train_loss": train_loss,
                "val_loss": val_loss
            }, (fname+".pth"))

            print(
                f"Best model saved: {(fname+'.pth')} "
                f"(val_acc={best_val_accuracy:.4f})"
            )

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stop_patience:
            print(
                f"Early stopping. "
                f"Best val accuracy: {best_val_accuracy:.4f}"
            )
            break

    return best_val_accuracy

# load pretrained model

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1

pt_model = resnet18(weights=weights)

num_classes = len(train_dataset.classes)

in_features = pt_model.fc.in_features


pt_model.fc = nn.Linear(
    pt_model.fc.in_features,
    num_classes
)


##
early_parameters = [
    parameter
    for module in (
        pt_model.conv1,
        pt_model.bn1,
        pt_model.layer1,
        pt_model.layer2
    )
    for parameter in module.parameters()
]
## up
pt_model = pt_model.to(device)

pt_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

pt_optimizer = torch.optim.AdamW([
    {
        "params": early_parameters,
        "lr": 3e-6
    },
    {
        "params": pt_model.layer3.parameters(),
        "lr": 1e-5
    },
    {
        "params": pt_model.layer4.parameters(),
        "lr": 3e-5
    },
    {
        "params": pt_model.fc.parameters(),
        "lr": 1e-5
    }
], weight_decay=5e-4)


pt_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    pt_optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    threshold=0.0005,
    threshold_mode="rel",
    cooldown=1,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
pt_model_filename = f"inat_resnet18_{num_classes}_ptclasses_imagenet"

training(50,pt_model,pt_optimizer,pt_scheduler,pt_criterion,pt_model_filename,train_dataset.classes)


Epoch 01/50 | Train loss: 5.9279 | Train acc: 0.0449 | Val loss: 5.3998 | Val acc: 0.1460 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet18_500_ptclasses_imagenet.pth (val_acc=0.1460)
Epoch 02/50 | Train loss: 5.0997 | Train acc: 0.2332 | Val loss: 4.7584 | Val acc: 0.2888 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet18_500_ptclasses_imagenet.pth (val_acc=0.2888)
Epoch 03/50 | Train loss: 4.5671 | Train acc: 0.3734 | Val loss: 4.3318 | Val acc: 0.3732 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet18_500_ptclasses_imagenet.pth (val_acc=0.3732)
Epoch 04/50 | Train loss: 4.1591 | Train acc: 0.4647 | Val loss: 3.9670 | Val acc: 0.4398 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet18_500_ptclasses_imagenet.pth (val_acc=0.4398)
Epoch 05/50 | Train loss: 3.8180 | Train acc: 0.5372 | Val loss: 3.7069 | Val acc: 0.4896 | LR: ['1.50e-06', '5.00e-

0.6346

In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2

pt_model = resnet50(weights=weights)

num_classes = len(train_dataset.classes)

in_features = pt_model.fc.in_features


pt_model.fc = nn.Linear(
    pt_model.fc.in_features,
    num_classes
)


##
early_parameters = [
    parameter
    for module in (
        pt_model.conv1,
        pt_model.bn1,
        pt_model.layer1,
        pt_model.layer2
    )
    for parameter in module.parameters()
]
## up
pt_model = pt_model.to(device)

pt_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

pt_optimizer = torch.optim.AdamW([
    {
        "params": early_parameters,
        "lr": 3e-6
    },
    {
        "params": pt_model.layer3.parameters(),
        "lr": 1e-5
    },
    {
        "params": pt_model.layer4.parameters(),
        "lr": 3e-5
    },
    {
        "params": pt_model.fc.parameters(),
        "lr": 1e-5
    }
], weight_decay=5e-4)



pt_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    pt_optimizer,
    mode="min",
    factor=0.5,             
    patience=3,             
    threshold=0.0005,
    threshold_mode="rel",
    cooldown=1,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
pt_model_filename = f"inat_resnet50_{num_classes}_ptclasses_imagenet"

training(50,pt_model,pt_optimizer,pt_scheduler,pt_criterion,pt_model_filename,train_dataset.classes)


Epoch 01/50 | Train loss: 5.9475 | Train acc: 0.0481 | Val loss: 5.2078 | Val acc: 0.1734 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet50_500_ptclasses_imagenet.pth (val_acc=0.1734)
Epoch 02/50 | Train loss: 4.6495 | Train acc: 0.2918 | Val loss: 3.8066 | Val acc: 0.4180 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet50_500_ptclasses_imagenet.pth (val_acc=0.4180)
Epoch 03/50 | Train loss: 3.5627 | Train acc: 0.5015 | Val loss: 3.0900 | Val acc: 0.5476 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet50_500_ptclasses_imagenet.pth (val_acc=0.5476)
Epoch 04/50 | Train loss: 2.8801 | Train acc: 0.6335 | Val loss: 2.6232 | Val acc: 0.6360 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet50_500_ptclasses_imagenet.pth (val_acc=0.6360)
Epoch 05/50 | Train loss: 2.4491 | Train acc: 0.7156 | Val loss: 2.3820 | Val acc: 0.6802 | LR: ['1.50e-06', '5.00e-

0.7794

# train with no preset weights

In [ ]:
np_model = resnet18(weights=None)

num_classes = len(train_dataset.classes)

np_model.fc = nn.Linear(
    np_model.fc.in_features,
    num_classes
)

np_model = np_model.to(device)

np_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

np_optimizer = torch.optim.AdamW(
    np_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)



np_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    np_optimizer,
    mode="max",       
    factor=0.5,      
    patience=2,       
    threshold=1e-3,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
np_model_filename = f"inat_resnet18_{num_classes}_npclasses_imagenet"

training(50,np_model,np_optimizer,np_scheduler,np_criterion,np_model_filename,train_dataset.classes)

Epoch 01/50 | Train loss: 5.9864 | Train acc: 0.0127 | Val loss: 5.7351 | Val acc: 0.0308 | LR: ['1.00e-04']
Best model saved: inat_resnet18_500_npclasses_imagenet.pth (val_acc=0.0308)
Epoch 02/50 | Train loss: 5.5809 | Train acc: 0.0328 | Val loss: 5.5716 | Val acc: 0.0350 | LR: ['1.00e-04']
Best model saved: inat_resnet18_500_npclasses_imagenet.pth (val_acc=0.0350)
Epoch 03/50 | Train loss: 5.3457 | Train acc: 0.0520 | Val loss: 5.3405 | Val acc: 0.0586 | LR: ['1.00e-04']
Best model saved: inat_resnet18_500_npclasses_imagenet.pth (val_acc=0.0586)
Epoch 04/50 | Train loss: 5.1474 | Train acc: 0.0718 | Val loss: 5.1827 | Val acc: 0.0772 | LR: ['1.00e-04']
Best model saved: inat_resnet18_500_npclasses_imagenet.pth (val_acc=0.0772)
Epoch 05/50 | Train loss: 4.9612 | Train acc: 0.0969 | Val loss: 5.0052 | Val acc: 0.0934 | LR: ['1.00e-04']
Best model saved: inat_resnet18_500_npclasses_imagenet.pth (val_acc=0.0934)
Epoch 06/50 | Train loss: 4.7805 | Train acc: 0.1213 | Val loss: 4.8869 | V

0.377

In [ ]:

np_model = resnet50(weights=None)

num_classes = len(train_dataset.classes)

np_model.fc = nn.Linear(
    np_model.fc.in_features,
    num_classes
)

np_model = np_model.to(device)

np_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

np_optimizer = torch.optim.AdamW(
    np_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)



np_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    np_optimizer,
    mode="max",       
    factor=0.5,       
    patience=2,       
    threshold=1e-3,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
np_model_filename = f"inat_resnet50_{num_classes}_npclasses_imagenet"

training(50,np_model,np_optimizer,np_scheduler,np_criterion,np_model_filename,train_dataset.classes)

Epoch 01/50 | Train loss: 6.1521 | Train acc: 0.0056 | Val loss: 5.9385 | Val acc: 0.0110 | LR: ['1.00e-04']
Best model saved: inat_resnet50_500_npclasses_imagenet.pth (val_acc=0.0110)
Epoch 02/50 | Train loss: 5.7923 | Train acc: 0.0150 | Val loss: 5.6815 | Val acc: 0.0234 | LR: ['1.00e-04']
Best model saved: inat_resnet50_500_npclasses_imagenet.pth (val_acc=0.0234)
Epoch 03/50 | Train loss: 5.5982 | Train acc: 0.0272 | Val loss: 5.5641 | Val acc: 0.0360 | LR: ['1.00e-04']
Best model saved: inat_resnet50_500_npclasses_imagenet.pth (val_acc=0.0360)
Epoch 04/50 | Train loss: 5.4389 | Train acc: 0.0409 | Val loss: 5.4463 | Val acc: 0.0418 | LR: ['1.00e-04']
Best model saved: inat_resnet50_500_npclasses_imagenet.pth (val_acc=0.0418)
Epoch 05/50 | Train loss: 5.2819 | Train acc: 0.0510 | Val loss: 5.2585 | Val acc: 0.0612 | LR: ['1.00e-04']
Best model saved: inat_resnet50_500_npclasses_imagenet.pth (val_acc=0.0612)
Epoch 06/50 | Train loss: 5.1145 | Train acc: 0.0720 | Val loss: 5.2160 | V

0.3446

# model reloading

In [ ]:
# ==================== Resume training ====================

pt_model_filename= Path("inat_resnet50_500_ptclasses_imagenet.pth")
ADDITIONAL_EPOCHS = 20   
LR_FACTOR = 0.3          

checkpoint = torch.load(
    pt_model_filename,
    map_location=device,
    weights_only=True
)


assert checkpoint["classes"] == train_dataset.classes
assert checkpoint["num_classes"] == len(train_dataset.classes)

# recover
pt_model.load_state_dict(
    checkpoint["model_state_dict"]
)

pt_optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

pt_scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

# make sure optizmizer is right device
for state in pt_optimizer.state.values():
    for key, value in state.items():
        if torch.is_tensor(value):
            state[key] = value.to(device)

start_epoch = checkpoint["epoch"]
best_val_accuracy = checkpoint["best_val_accuracy"]


for group in pt_optimizer.param_groups:
    group["lr"] *= LR_FACTOR

total_epochs = start_epoch + ADDITIONAL_EPOCHS

print(f"Loaded: {pt_model_filename}")
print(f"Resume from epoch: {start_epoch + 1}")
print(f"Train until epoch: {total_epochs}")
print(f"Previous best val accuracy: {best_val_accuracy:.4f}")
print("Learning rates:", [
    group["lr"]
    for group in pt_optimizer.param_groups
])

best_accuracy = training(
    total_epochs,
    pt_model,
    pt_optimizer,
    pt_scheduler,
    pt_criterion,
    pt_model_filename,
    train_dataset.classes,
)

Loaded: inat_resnet50_500_ptclasses_imagenet.pth
Resume from epoch: 31
Train until epoch: 50
Previous best val accuracy: 0.7822
Learning rates: [3e-08, 4.6875e-08, 1.40625e-07, 4.6875e-08]
Epoch 01/50 | Train loss: 1.3966 | Train acc: 0.9464 | Val loss: 1.8818 | Val acc: 0.7810 | LR: ['3.00e-08', '4.69e-08', '1.41e-07', '4.69e-08']


TypeError: unsupported operand type(s) for +: 'WindowsPath' and 'str'